# vForge — 03. Fine-tune Gemma-2-2B on GPU (LoRA, PyTorch + PEFT)

GPU baseline for the TPU comparison. Colab → Runtime → T4 (free) or A100 / L4 (Pro).

## 1. Install

In [ ]:
!pip -q install -U torch transformers peft trl datasets accelerate bitsandbytes huggingface_hub

## 2. Verify GPU

In [ ]:
import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(f'{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 3. Load + format dataset (mirror notebook 02)

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files='data/code.jsonl', split='train')
def fmt(r):
    p = f'<start_of_turn>user\n{r["instruction"]}'
    if r.get('input'): p += f'\n\n{r["input"]}'
    p += f'<end_of_turn>\n<start_of_turn>model\n{r["output"]}<end_of_turn>'
    return {'text': p}
ds = ds.map(fmt)
ds[0]['text'][:300]

## 4. Tokenize

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('google/gemma-2-2b')
if tok.pad_token is None: tok.pad_token = tok.eos_token
def t(b): return tok(b['text'], truncation=True, max_length=512, padding='max_length')
ds = ds.map(t, batched=True, remove_columns=['text','instruction','input','output'])
ds.set_format('torch', columns=['input_ids','attention_mask'])

## 5. Load model + apply LoRA

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    'google/gemma-2-2b', torch_dtype=torch.bfloat16, device_map='auto')
print(f'load: {time.time()-t0:.1f}s')
lora = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
                  task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj'])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

## 6. Train

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
torch.cuda.reset_peak_memory_stats()
args = TrainingArguments(output_dir='out_gpu',
  per_device_train_batch_size=4, gradient_accumulation_steps=4,
  num_train_epochs=1, learning_rate=1e-4, bf16=True,
  logging_steps=10, save_strategy='no', warmup_ratio=0.03,
  report_to=[])
tr = Trainer(model=model, args=args, train_dataset=ds,
             data_collator=DataCollatorForLanguageModeling(tokenizer=tok, mlm=False))
import time, json
t0 = time.time()
out = tr.train()
elapsed = time.time() - t0
metrics = {
  'hardware':'gpu', 'device': torch.cuda.get_device_name(0),
  'base_model':'google/gemma-2-2b', 'rank':8,
  'epochs':1, 'batch_size':4,
  'dataset_rows': len(ds),
  'train_time_sec': round(elapsed,2),
  'final_loss': float(out.training_loss),
  'peak_memory_gb': round(torch.cuda.max_memory_allocated()/1e9, 3),
  'approx_train_tokens_per_sec': round(len(ds) * 512 / elapsed, 2),
}
print(json.dumps(metrics, indent=2))

## 7. Save

In [ ]:
from pathlib import Path
Path('out_gpu').mkdir(exist_ok=True)
model.save_pretrained('out_gpu')
tok.save_pretrained('out_gpu')
Path('out_gpu/metrics.json').write_text(json.dumps(metrics, indent=2))